In [191]:
# importando bibliotecas
from faker import Faker
from random import choice, choices, randint, triangular, uniform
import pandas as pd
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from unicodedata import normalize
from re import sub, IGNORECASE

In [192]:
# Carregando variáveis de ambiente
load_dotenv()

# Acessando o banco de dados
DB_URL = os.getenv("DB_URL")
engine = create_engine(DB_URL)
faker = Faker('pt_BR')

In [193]:
def apenas_numeros(valor: str) -> str:
    return sub(r'\D', '', valor)

def remove_acentos(texto: str) -> str:
    return normalize('NFKD', texto.lower()).encode('ASCII', 'ignore').decode('ASCII')

In [194]:
# Constantes
QTD_LOJAS               = 10
QTD_BANCOS              = 5
QTD_CONTAS              = 80
QTD_PAGAMENTOS          = 1000
QTD_VENDAS              = 1000
QTD_TRANSPORTADORAS     = 8
QTD_FUNCIONARIOS        = 50
QTD_ENVIA_ITENS         = 3000
QTD_CONTA_LOJA          = 15
QTD_CONTA_FUNCIONARIO   = 60
QTD_PAGAMENTO_PARCELADO = 300

In [195]:
dados_loja = []
for i in range(QTD_LOJAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_loja.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'email': faker.email(),
        'telefone': telefone_formatado,
        'percentual_comissao': round(uniform(0.01, 1), 2)
    })

df_lojas = pd.DataFrame(dados_loja)
display(df_lojas.head())

,nome,cnpj,email,telefone,percentual_comissao
0,Siqueira,31746958000150,marinamonteiro@example.com,5526905448430,0.13
1,Camargo,94128075000104,hellenacorreia@example.com,5572999621000,0.53
2,Cassiano e Filhos,13720695000128,lizmendes@example.net,55000992976599,0.61
3,Camargo Nogueira e Filhos,52319674000157,castroisabella@example.org,5526916683375,0.14
4,Guerra - EI,04298713000134,rafaelanogueira@example.org,55011911294895,0.02


In [196]:
dados_banco = []
for i in range(QTD_BANCOS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    
    dados_banco.append({
        'nome': f'Banco {faker.company()}',
        'cnpj': cnpj_formatado
    })

df_bancos = pd.DataFrame(dados_banco)
display(df_bancos.head())

,nome,cnpj
0,Banco Monteiro Fernandes e Filhos,63947825000138
1,Banco das Neves,75869123000162
2,Banco Garcia Cassiano - EI,15234967000105
3,Banco Albuquerque Cavalcante - ME,17623485000172
4,Banco Gomes,65349271000100


In [197]:
dados_conta = []

TIPOS_CONTA = ['CORRENTE', 'POUPANCA', 'SALARIO']

for i in range(QTD_CONTAS):
    dados_conta.append({
        'agencia': str(randint(1, 9999)).zfill(4),
        'numero_conta': str(randint(1, 999999)).zfill(6),
        'tipo_conta': choice(TIPOS_CONTA),
        'saldo': round(triangular(0, 9999.99, 100), 2),
        'cod_banco': randint(1, QTD_BANCOS)
    })

df_contas = pd.DataFrame(dados_conta)
display(df_contas.head())

,agencia,numero_conta,tipo_conta,saldo,cod_banco
0,1106,993630,POUPANCA,2532.64,4
1,6603,608405,CORRENTE,2316.78,5
2,0059,718968,POUPANCA,1042.19,2
3,4348,224593,SALARIO,8879.32,4
4,0382,940866,SALARIO,6556.22,2


In [198]:
dados_pagamento = []

TIPOS_PAGAMENTO = ['BOLETO', 'PIX', 'DEBITO', 'CREDITO']
STATUS_PAGAMENTO = ['AGUARDANDO', 'CONCLUIDO', 'EXPIRADO']

for i in range(QTD_PAGAMENTOS):
    cod_conta_origem = randint(1, QTD_CONTAS)
    cod_conta_destino = randint(1, QTD_CONTAS)
    while cod_conta_destino == cod_conta_origem:
        cod_conta_destino = randint(1, QTD_CONTAS)
    
    dados_pagamento.append({
        'tipo_pagamento': choice(TIPOS_PAGAMENTO),
        'status_pagamento': choice(STATUS_PAGAMENTO),
        'valor': round(triangular(0, 999.99, 100), 2),
        'cod_conta_origem': cod_conta_origem,
        'cod_conta_destino': cod_conta_destino
    })

df_pagamentos = pd.DataFrame(dados_pagamento)
display(df_pagamentos.head())

,tipo_pagamento,status_pagamento,valor,cod_conta_origem,cod_conta_destino
0,PIX,CONCLUIDO,229.30,50,7
1,DEBITO,EXPIRADO,325.27,1,47
2,CREDITO,CONCLUIDO,150.24,69,56
3,DEBITO,CONCLUIDO,60.38,68,41
4,DEBITO,AGUARDANDO,154.61,40,26


In [199]:
dados_venda = []

STATUS_VENDA = ['APROVADO', 'CANCELADO', 'ESTORNADO']

for i in range(QTD_VENDAS):    
    dados_venda.append({
        'valor_total': round(triangular(0, 999.99, 100), 2),
        'status_venda': choices(STATUS_VENDA, weights=[85, 5, 10], k=1)[0],
        'cod_vendedor': randint(1, QTD_FUNCIONARIOS)
    })

df_vendas = pd.DataFrame(dados_venda)
display(df_vendas.head())

,valor_total,status_venda,cod_vendedor
0,289.46,APROVADO,18
1,79.52,APROVADO,37
2,66.72,APROVADO,17
3,298.55,APROVADO,18
4,127.55,APROVADO,15


In [200]:
dados_transportadora = []
for i in range(QTD_TRANSPORTADORAS):
    cnpj_formatado = apenas_numeros(faker.cnpj())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    
    dados_transportadora.append({
        'nome': faker.company(),
        'cnpj': cnpj_formatado,
        'telefone': telefone_formatado,
        'email': faker.email(),
        'taxa_entrega': round(uniform(0.01, 1), 2)
    })

df_transportadoras = pd.DataFrame(dados_transportadora)
display(df_transportadoras.head())

,nome,cnpj,telefone,email,taxa_entrega
0,Ferreira,06715984000173,5541980434724,barbosathiago@example.net,0.23
1,Farias,96347205000117,5514900526255,yabreu@example.com,0.37
2,Pacheco Caldeira e Filhos,42016739000131,5500924420891,valentinacavalcanti@example.net,0.02
3,da Paz Rios e Filhos,50647819000113,5575958970595,sophie62@example.net,0.47
4,Machado,26905471000132,5544958956837,ana-carolinacasa-grande@example.net,0.28


In [201]:
dados_funcionario = []
for i in range(QTD_FUNCIONARIOS):
    nome = sub(r'\b(sr|sra|srta|dr|dra)\.\s*', '', faker.name(), flags=IGNORECASE)
    cpf_formatado = apenas_numeros(faker.cpf())
    telefone_formatado = apenas_numeros(faker.cellphone_number())
    email = f'{sub(r' ', '_', remove_acentos(nome))}@gmail.com'
    
    dados_funcionario.append({
        'nome': nome,
        'cpf': cpf_formatado,
        'cargo': faker.job(),
        'salario': round(triangular(1621, 4000, 100), 2),
        'email': email,
        'telefone': telefone_formatado,
        'cod_loja': randint(1, QTD_LOJAS)
    })

df_funcionarios = pd.DataFrame(dados_funcionario)
display(df_funcionarios.head())

,nome,cpf,cargo,salario,email,telefone,cod_loja
0,Léo Teixeira,52809413606,Ambientalista,3049.73,leo_teixeira@gmail.com,55089946162102,7
1,Thomas Sampaio,03256841970,Piloto automobilístico,2092.59,thomas_sampaio@gmail.com,5553980099918,7
2,Beatriz Ribeiro,61708435948,Datilógrafo,2276.28,beatriz_ribeiro@gmail.com,5586935014706,5
3,Antonella Pereira,17423895032,Cambista,2587.97,antonella_pereira@gmail.com,5586904566937,8
4,Daniel Pastor,76854013984,Taxista,1674.65,daniel_pastor@gmail.com,5587948225067,9


In [202]:
dados_envia_itens = []
dados_conta_loja = []
dados_conta_funcionario = []
dados_pagamento_parcelado = []

for i in range(QTD_ENVIA_ITENS):
    dados_envia_itens.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_transportadora': randint(1, QTD_TRANSPORTADORAS)
    })

for i in range(QTD_CONTA_LOJA):
    dados_conta_loja.append({
        'cod_loja': randint(1, QTD_LOJAS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_CONTA_FUNCIONARIO):
    dados_conta_funcionario.append({
        'cod_funcionario': randint(1, QTD_FUNCIONARIOS),
        'cod_conta': randint(1, QTD_CONTAS)
    })

for i in range(QTD_PAGAMENTO_PARCELADO):
    dados_pagamento_parcelado.append({
        'cod_pagamento': randint(1, QTD_PAGAMENTOS),
        'cod_venda': randint(1, QTD_VENDAS)
    })

df_envia_itens = pd.DataFrame(dados_envia_itens)
df_conta_loja = pd.DataFrame(dados_conta_loja)
df_conta_funcionario = pd.DataFrame(dados_conta_funcionario)
df_pagamento_parcelado = pd.DataFrame(dados_pagamento_parcelado)

In [ ]:
# Inserindo no banco de dados
tabelas = {
    'tb_loja': df_lojas,
    'tb_banco': df_bancos,
    'tb_transportadora': df_transportadoras,
    'tb_funcionario': df_funcionarios,
    'tb_conta': df_contas,
    'tb_pagamento': df_pagamentos,
    'tb_venda': df_vendas,
    'tb_envia_itens': df_envia_itens,
    'tb_conta_loja': df_conta_loja,
    'tb_conta_funcionario': df_conta_funcionario,
    'tb_pagamento_parcelado': df_pagamento_parcelado
}

print('Iniciando conexão com o banco de dados')
for t, df in tabelas.items():
    try:
        df.to_sql(name=t, con=engine, if_exists='append', index=False)
    except Exception as e:
        print(f'Um erro no banco aconteceu')
        print(e)
    print('Conexão feita com sucesso')

Iniciando conexão com o banco de dados
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
Conexão feita com sucesso
